#### Imports & Install

In [1]:
import psycopg2
import numpy as np
import pandas as pd

#### Reading the csv file

In [9]:
retail_df = pd.read_csv(r'raw_data\Sales Dataset .csv')

#### Inspecting data for anomalies

In [93]:
retail_df.columns

Index(['Order ID', 'Amount', 'Profit', 'Quantity', 'Category', 'Sub-Category',
       'PaymentMode', 'Order Date', 'CustomerName', 'State', 'City',
       'Year-Month'],
      dtype='object')

In [10]:
retail_df.columns = retail_df.columns.str.strip().str.lower().str.replace(' ', '_')
retail_df.columns

Index(['order_id', 'amount', 'profit', 'quantity', 'category', 'sub-category',
       'paymentmode', 'order_date', 'customername', 'state', 'city',
       'year-month'],
      dtype='object')

In [11]:
retail_df.rename(columns={'sub-category': 'sub_category', 
                          'paymentmode': 'payment_mode', 
                          'customername': 'customer_name', 
                          'year-month': 'year_month'}, inplace=True)
retail_df.columns

Index(['order_id', 'amount', 'profit', 'quantity', 'category', 'sub_category',
       'payment_mode', 'order_date', 'customer_name', 'state', 'city',
       'year_month'],
      dtype='object')

In [6]:
retail_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194 entries, 0 to 1193
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   order_id       1194 non-null   object
 1   amount         1194 non-null   int64 
 2   profit         1194 non-null   int64 
 3   quantity       1194 non-null   int64 
 4   category       1194 non-null   object
 5   sub_category   1194 non-null   object
 6   payment_mode   1194 non-null   object
 7   order_date     1194 non-null   object
 8   customer_name  1194 non-null   object
 9   state          1194 non-null   object
 10  city           1194 non-null   object
 11  year_month     1194 non-null   object
dtypes: int64(3), object(9)
memory usage: 112.1+ KB


In [12]:
retail_df['order_date'] = pd.to_datetime(retail_df['order_date'])
retail_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194 entries, 0 to 1193
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       1194 non-null   object        
 1   amount         1194 non-null   int64         
 2   profit         1194 non-null   int64         
 3   quantity       1194 non-null   int64         
 4   category       1194 non-null   object        
 5   sub_category   1194 non-null   object        
 6   payment_mode   1194 non-null   object        
 7   order_date     1194 non-null   datetime64[ns]
 8   customer_name  1194 non-null   object        
 9   state          1194 non-null   object        
 10  city           1194 non-null   object        
 11  year_month     1194 non-null   object        
dtypes: datetime64[ns](1), int64(3), object(8)
memory usage: 112.1+ KB


In [13]:
retail_df = retail_df.drop(columns=['year_month'])
retail_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194 entries, 0 to 1193
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       1194 non-null   object        
 1   amount         1194 non-null   int64         
 2   profit         1194 non-null   int64         
 3   quantity       1194 non-null   int64         
 4   category       1194 non-null   object        
 5   sub_category   1194 non-null   object        
 6   payment_mode   1194 non-null   object        
 7   order_date     1194 non-null   datetime64[ns]
 8   customer_name  1194 non-null   object        
 9   state          1194 non-null   object        
 10  city           1194 non-null   object        
dtypes: datetime64[ns](1), int64(3), object(7)
memory usage: 102.7+ KB


In [15]:
customer_df_main = retail_df[['customer_name']].drop_duplicates().reset_index(drop=True)
customer_df_main['customer_id'] = customer_df_main.index + 1
customer_df_main = customer_df_main[['customer_id', 'customer_name']]
customer_df_main

,customer_id,customer_name
0,1,David Padilla
1,2,Connor Morgan
2,3,Robert Stone
3,4,John Fields
4,5,Clayton Smith
...,...,...
797,798,Megan Mclean
798,799,Caitlin Hunt
799,800,Jenna Holland
800,801,Stephanie Oconnell


In [16]:
retail_df = pd.merge(retail_df, customer_df_main[['customer_id','customer_name']], on=['customer_name'], how='left')
retail_df

,order_id,amount,profit,quantity,category,sub_category,payment_mode,order_date,customer_name,state,city,customer_id
0,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2023-06-27,David Padilla,Florida,Miami,1
1,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2024-12-27,Connor Morgan,Illinois,Chicago,2
2,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2021-07-25,Robert Stone,New York,Buffalo,3
3,B-26776,4975,1330,14,Electronics,Printers,UPI,2023-06-27,David Padilla,Florida,Miami,1
4,B-26776,4975,1330,14,Electronics,Printers,UPI,2024-12-27,Connor Morgan,Illinois,Chicago,2
...,...,...,...,...,...,...,...,...,...,...,...,...
1189,B-26370,8825,3594,15,Furniture,Tables,Debit Card,2024-07-31,Megan Mclean,New York,New York City,798
1190,B-26298,2082,642,8,Electronics,Phones,EMI,2020-06-02,Caitlin Hunt,New York,Rochester,799
1191,B-26298,2082,642,8,Electronics,Phones,EMI,2022-12-15,Jenna Holland,Texas,Austin,800
1192,B-26298,2082,642,8,Electronics,Phones,EMI,2020-08-07,Stephanie Oconnell,New York,Buffalo,801


In [53]:
customer_df = retail_df[[ 'customer_id', 'customer_name', 'state', 'city']].reset_index(drop=True)
customer_df['customer_pk'] = customer_df.index + 1
customer_df = customer_df[['customer_pk', 'customer_id', 'customer_name', 'state', 'city']]
customer_df

,customer_pk,customer_id,customer_name,state,city
0,1,1,David Padilla,Florida,Miami
1,2,2,Connor Morgan,Illinois,Chicago
2,3,3,Robert Stone,New York,Buffalo
3,4,1,David Padilla,Florida,Miami
4,5,2,Connor Morgan,Illinois,Chicago
...,...,...,...,...,...
1189,1190,798,Megan Mclean,New York,New York City
1190,1191,799,Caitlin Hunt,New York,Rochester
1191,1192,800,Jenna Holland,Texas,Austin
1192,1193,801,Stephanie Oconnell,New York,Buffalo


In [20]:
payment = retail_df[['payment_mode']].drop_duplicates().reset_index(drop=True)
payment['payment_id'] = payment.index + 1
payment = payment[['payment_id', 'payment_mode']]
payment

,payment_id,payment_mode
0,1,UPI
1,2,Debit Card
2,3,EMI
3,4,Credit Card
4,5,COD


In [21]:
retail_df = pd.merge(retail_df, payment[['payment_id', 'payment_mode']], on=['payment_mode'], how='left')
retail_df

,order_id,amount,profit,quantity,category,sub_category,payment_mode,order_date,customer_name,state,city,customer_id,payment_id
0,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2023-06-27,David Padilla,Florida,Miami,1,1
1,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2024-12-27,Connor Morgan,Illinois,Chicago,2,1
2,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2021-07-25,Robert Stone,New York,Buffalo,3,1
3,B-26776,4975,1330,14,Electronics,Printers,UPI,2023-06-27,David Padilla,Florida,Miami,1,1
4,B-26776,4975,1330,14,Electronics,Printers,UPI,2024-12-27,Connor Morgan,Illinois,Chicago,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1189,B-26370,8825,3594,15,Furniture,Tables,Debit Card,2024-07-31,Megan Mclean,New York,New York City,798,2
1190,B-26298,2082,642,8,Electronics,Phones,EMI,2020-06-02,Caitlin Hunt,New York,Rochester,799,3
1191,B-26298,2082,642,8,Electronics,Phones,EMI,2022-12-15,Jenna Holland,Texas,Austin,800,3
1192,B-26298,2082,642,8,Electronics,Phones,EMI,2020-08-07,Stephanie Oconnell,New York,Buffalo,801,3


In [54]:
order_df = retail_df[['order_id', 'order_date', 'payment_id']].reset_index(drop=True)
order_df['order_pk'] = order_df.index + 1
order_df['customer_pk'] = customer_df['customer_pk']
order_df = order_df[['order_pk', 'order_id', 'order_date','customer_pk', 'payment_id']]
order_df

,order_pk,order_id,order_date,customer_pk,payment_id
0,1,B-26776,2023-06-27,1,1
1,2,B-26776,2024-12-27,2,1
2,3,B-26776,2021-07-25,3,1
3,4,B-26776,2023-06-27,4,1
4,5,B-26776,2024-12-27,5,1
...,...,...,...,...,...
1189,1190,B-26370,2024-07-31,1190,2
1190,1191,B-26298,2020-06-02,1191,3
1191,1192,B-26298,2022-12-15,1192,3
1192,1193,B-26298,2020-08-07,1193,3


In [22]:
product_category_df = retail_df[['category', 'sub_category']].drop_duplicates().reset_index(drop=True)
product_category_df['category_id'] = product_category_df.index + 1
product_category_df = product_category_df[['category_id', 'category', 'sub_category']]
product_category_df

,category_id,category,sub_category
0,1,Electronics,Electronic Games
1,2,Electronics,Printers
2,3,Office Supplies,Pens
3,4,Electronics,Laptops
4,5,Furniture,Tables
5,6,Furniture,Chairs
6,7,Office Supplies,Markers
7,8,Furniture,Sofas
8,9,Office Supplies,Paper
9,10,Office Supplies,Binders


In [23]:
retail_df = pd.merge(retail_df, product_category_df[['category_id', 'sub_category']], on=['sub_category'], how='left')
retail_df

,order_id,amount,profit,quantity,category,sub_category,payment_mode,order_date,customer_name,state,city,customer_id,payment_id,category_id
0,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2023-06-27,David Padilla,Florida,Miami,1,1,1
1,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2024-12-27,Connor Morgan,Illinois,Chicago,2,1,1
2,B-26776,9726,1275,5,Electronics,Electronic Games,UPI,2021-07-25,Robert Stone,New York,Buffalo,3,1,1
3,B-26776,4975,1330,14,Electronics,Printers,UPI,2023-06-27,David Padilla,Florida,Miami,1,1,2
4,B-26776,4975,1330,14,Electronics,Printers,UPI,2024-12-27,Connor Morgan,Illinois,Chicago,2,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1189,B-26370,8825,3594,15,Furniture,Tables,Debit Card,2024-07-31,Megan Mclean,New York,New York City,798,2,5
1190,B-26298,2082,642,8,Electronics,Phones,EMI,2020-06-02,Caitlin Hunt,New York,Rochester,799,3,11
1191,B-26298,2082,642,8,Electronics,Phones,EMI,2022-12-15,Jenna Holland,Texas,Austin,800,3,11
1192,B-26298,2082,642,8,Electronics,Phones,EMI,2020-08-07,Stephanie Oconnell,New York,Buffalo,801,3,11


In [ ]:
retail_df = pd.merge(retail_df, order_df[['order_id', 'order_pk', 'customer_id']], on=['order_id'], how='left')
retail_df

In [41]:
order_details_df = retail_df[['category_id','amount', 'profit', 'quantity']].reset_index(drop=True)
order_details_df['order_pk']= order_df['order_pk']
order_details_df['order_details_id'] = order_details_df.index + 1
order_details_df = order_details_df[['order_details_id', 'order_pk', 'category_id','amount', 'profit', 'quantity' ]]
order_details_df

,order_details_id,order_pk,category_id,amount,profit,quantity
0,1,1,1,9726,1275,5
1,2,2,1,9726,1275,5
2,3,3,1,9726,1275,5
3,4,4,2,4975,1330,14
4,5,5,2,4975,1330,14
...,...,...,...,...,...,...
1189,1190,1190,5,8825,3594,15
1190,1191,1191,11,2082,642,8
1191,1192,1192,11,2082,642,8
1192,1193,1193,11,2082,642,8


#### Aggregates and filters

In [ ]:
#Aggregrates monthly profit
retail_df['Month'] = retail_df['order_date'].dt.month_name()
monthly_profits = retail_df.groupby('Month')['profit'].sum().reset_index()
monthly_profits.sort_values(by='profit', ascending=False).reset_index(drop=True)

In [289]:
order_detail= pd.merge(order_df, order_details_df, on='order_id')


In [290]:
customer_orders= pd.merge(order_detail, customer_df, on='customer_id')

In [ ]:
#Cities with low profit
cities_low_profit = customer_orders.groupby('city', as_index=False)['profit'].sum().sort_values(by= 'profit', ascending=True).reset_index(drop=True)
cities_low_profit

In [ ]:
# Top purchasing customers
top_customers = customer_orders.groupby('customer_id', as_index=False)['amount'].sum().sort_values(by='amount', ascending=False).reset_index(drop=True)
top_customers_twenty =top_customers.head(20)
top_customers_twenty

In [ ]:
#Quantity sold per category
category_order = pd.merge(order_details_df, product_category_df, on= 'category_id')
quantity_category = category_order.groupby('category_id', as_index =False)['quantity'].sum().sort_values(by='quantity', ascending=False).reset_index(drop=True)
quantity_category

#### Converting data csv files(normal and aggregates)

In [55]:
customer_df.to_csv(r'clean_data/customer.csv', index=False)
order_df.to_csv(r'clean_data/order.csv', index=False)
order_details_df.to_csv(r'clean_data/order_details.csv', index=False)
product_category_df.to_csv(r'clean_data/product_category.csv', index=False)



In [296]:
quantity_category.to_csv(r'aggregated_data/quantity_category.csv', index=False)
cities_low_profit.to_csv(r'aggregated_data/cities_low_profit.csv', index=False)
top_customers_twenty.to_csv(r'aggregated_data/top_customers_twenty.csv', index=False)
monthly_profits.to_csv(r'aggregated_data/monthly_profits.csv', index=False)


#### Data loading

In [37]:
def db_connection():
    connection = psycopg2.connect(
        host="localhost",
        user="postgres",
        database="capstone_retail",
        password= "Cisco123"
    )
    return connection

In [38]:
conn = db_connection()



In [34]:
customer_df.columns

Index(['customer_pk', 'customer_id', 'customer_name', 'state', 'city'], dtype='object')

In [56]:
def create_tables():
    conn = db_connection()
    cursor = conn.cursor()
    create_table_query = ''' 
                            CREATE SCHEMA IF NOT EXISTS retail;
                            
                            DROP TABLE IF EXISTS retail.customer CASCADE;
                            DROP TABLE IF EXISTS retail.order CASCADE;
                            DROP TABLE IF EXISTS retail.order_details CASCADE;
                            DROP TABLE IF EXISTS retail.product_category CASCADE;
                            
                            CREATE TABLE IF NOT EXISTS retail.customer (
                                customer_pk INTEGER PRIMARY KEY,
                                customer_id INTEGER,
                                customer_name TEXT,
                                state TEXT,
                                city TEXT
                            );

                            CREATE TABLE IF NOT EXISTS retail.order (
                                order_pk INTEGER PRIMARY KEY,
                                order_id TEXT,
                                order_date DATE,
                                customer_pk INTEGER,
                                payment_id INTEGER,
                                FOREIGN KEY (customer_pk) references retail.customer(customer_pk)
                            );
                            
                            CREATE TABLE IF NOT EXISTS retail.product_category (
                                category_id INTEGER PRIMARY KEY,
                                category TEXT,
                                sub_category TEXT
                            );
                            
                            CREATE TABLE IF NOT EXISTS retail.order_details (
                                order_details_id INTEGER PRIMARY KEY,
                                order_pk INTEGER,
                                category_id INTEGER,
                                amount INTEGER,
                                profit INTEGER,
                                quantity INTEGER,
                                FOREIGN KEY (order_pk) references retail.order(order_pk),
                                FOREIGN KEY (category_id) references retail.product_category(category_id)
                            );'''
                            
    cursor.execute(create_table_query)
    conn.commit()
    cursor.close()
    conn.close()


In [58]:
create_tables()

In [59]:
import csv
def load_data_from_csv(csv_path):
    conn = db_connection()
    cursor = conn.cursor()
    with open(csv_path, 'r') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute(''' 
                           INSERT INTO retail.customer (customer_pk, customer_id, customer_name, state, city)
                           VALUES (%s, %s, %s, %s, %s);''',
                           row  
                        ) 
    conn.commit()
    cursor.close()
    conn.close()
    
csv_file_path = r'clean_data\customer.csv'
load_data_from_csv(csv_file_path)
    

In [60]:
import csv
def load_data_from_csv(csv_path):
    conn = db_connection()
    cursor = conn.cursor()
    with open(csv_path, 'r') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute(''' 
                           INSERT INTO retail.order (order_pk, order_id, order_date, customer_pk, payment_id)
                           VALUES (%s, %s, %s, %s, %s);''',
                           row  
                        ) 
    conn.commit()
    cursor.close()
    conn.close()
    
csv_file_path = r'clean_data\order.csv'
load_data_from_csv(csv_file_path)

In [61]:
import csv
def load_data_from_csv(csv_path):
    conn = db_connection()
    cursor = conn.cursor()
    with open(csv_path, 'r') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute(''' 
                           INSERT INTO retail.product_category (category_id, category, sub_category)
                           VALUES (%s, %s, %s);''',
                           row  
                        ) 
    conn.commit()
    cursor.close()
    conn.close()
    
csv_file_path = r'clean_data\product_category.csv'
load_data_from_csv(csv_file_path)

In [62]:
import csv
def load_data_from_csv(csv_path):
    conn = db_connection()
    cursor = conn.cursor()
    with open(csv_path, 'r') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:
            cursor.execute(''' 
                           INSERT INTO retail.order_details (order_details_id, order_pk, category_id, amount, profit, quantity)
                           VALUES (%s, %s, %s, %s, %s, %s);''',
                           row  
                        ) 
    conn.commit()
    cursor.close()
    conn.close()
    
csv_file_path = r'clean_data\order_details.csv'
load_data_from_csv(csv_file_path)